# Automatic online collection of annotated poems from For Better For Verse.

## 1. Library import

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time


## 2. Web page retrieval

In [ ]:
url1 = "https://prosody.lib.virginia.edu/" # the For Better For Verse website

# Defining a User-Agent header for HTTP requests
headers = {"User-Agent": "Mozilla/5.0"} 

#Retrieve the page containing the list of poems
response = requests.get(url1, headers=headers) 
if response.status_code != 200:
    raise RuntimeError(f'Problem with {url1}')


In [ ]:
# parsing with beautifulsoup
html1 = response.text
soup = BeautifulSoup(html1, "html.parser")
# poems links are in the "titles" tags
poems_links = soup.select_one(".titles")

'<!DOCTYPE html>\n<html lang="en">\n<head>\n    <meta charset ="UTF-8" />\n    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n    <title>For Better For Verse</title>\n    <meta name="description" content="">\n\n    <!-- Javascripts -->\n\n    <script type="text/javascript">var tb_pathToImage = "https://prosody.lib.virginia.edu/wp-content/themes/prosody_theme/images/loadingAnimation.gif";</script>\n\n    <!-- RSS feed, Atom feed, pingback url -->\n\n    <link rel="alternate" type="application/rss+xml" title="For Better For Verse RSS Feed" href="https://prosody.lib.virginia.edu/feed/" />\n    <link rel="alternate" type="application/atom+xml" title="Atom 0.3" href="https://prosody.lib.virginia.edu/feed/atom/" />\n    <link rel="pingback" href="https://prosody.lib.virginia.edu/xmlrpc.php" />\n\n    <!-- wp_head is necessary for a lot of wp hooks and actions, such as javascript queueing  -->\n    <meta name=\'robots\' content=\'max-image-preview:large\' />\n<style i

In [9]:
links = []
#retrieving all the links
for a in poems_links.find_all('a', href=True):
    link = a['href']
    links.append(link)


In [10]:
links

['https://prosody.lib.virginia.edu/prosody_poem/a-musical-instrument/',
 'https://prosody.lib.virginia.edu/prosody_poem/a-slumber-did-my-spirit-seal/',
 'https://prosody.lib.virginia.edu/prosody_poem/from-the-prelude-i-326-39/',
 'https://prosody.lib.virginia.edu/prosody_poem/adonais-stanzas-54-55/',
 'https://prosody.lib.virginia.edu/prosody_poem/ah-sun-flower/',
 'https://prosody.lib.virginia.edu/prosody_poem/amoretti-75/',
 'https://prosody.lib.virginia.edu/prosody_poem/an-essay-on-criticism-excerpt-11-362-73/',
 'https://prosody.lib.virginia.edu/prosody_poem/anthem-for-doomed-youth/',
 'https://prosody.lib.virginia.edu/prosody_poem/astrophil-and-stella-31/',
 'https://prosody.lib.virginia.edu/prosody_poem/beppo-stanza-xliv/',
 'https://prosody.lib.virginia.edu/prosody_poem/bright-star/',
 'https://prosody.lib.virginia.edu/prosody_poem/cary-morison-ode-the-turn-65-74/',
 'https://prosody.lib.virginia.edu/prosody_poem/chorus-from-henry-the-fifth-iii-i-1-17/',
 'https://prosody.lib.vi

## 3. Annotations retrieval

In [ ]:
lines_annotation =[]
j = 0
for link in links : # for each page (each poem)
    # retrieving a page
    response = requests.get(link, headers=headers)
    if response.status_code != 200:
        raise RuntimeError(f'Problem with {link}')
    html = response.text

    soup = BeautifulSoup(html, "html.parser")
    annotations = soup.select(".TEI-l") 
    # basic information
    information =  soup.select_one("#poemtitle")
    title = information.select_one("h2").get_text()
    author = information.select_one("h4").get_text()
    for i in annotations:
        data_annotation = {} 
        attributes = i.attrs
        data_annotation["line"] = attributes["data-feet"].replace("|", "") # getting rid of the foot separators
        if "data-met" in attributes.keys():
            data_annotation["met"] = attributes["data-met"] # the "ideal" stress sequence (following the main rythmical scheme of the poem perfectly)
        else :
            pass
        data_annotation["real"] = attributes["data-real"] # the real stress sequence
        data_annotation["feet"] = attributes["data-feet"]
        counts = attributes["data-feet"].count("|") + 1

        # number of feet
        if counts ==  1 : 
            data_annotation["feet_number"] = "monometer"
        elif counts == 2 :
            data_annotation["feet_nuber"] = "dimeter"
        elif counts == 3 :
            data_annotation["feet_number"] = "trimeter"
        elif counts == 4 :
            data_annotation["feet_number"] = "tetrameter"
        elif counts == 5 :
            data_annotation["feet_number"] = "pentameter"
        else : 
            data_annotation["feet_number"] = "hexameter"
        span_one = i.select_one("span")
        attrs_meter= dict(span_one.attrs)
        raw_meter_name = attrs_meter["answer"].split("(")[0]

        # foot label
        if raw_meter_name == "-+" :
            data_annotation["meter_quality"] = "iambic"
        elif raw_meter_name == "+-":
            data_annotation["meter_quality"] = "trochaic"
        elif raw_meter_name == "--+":
            data_annotation["meter_quality"] =  "anapestic"
        elif raw_meter_name == "+--":
            data_annotation["meter_quality"] = "dactylic"
        elif raw_meter_name == "++" : 
            data_annotation["meter_quality"] = "spondaic"
        else : 
            data_annotation["meter_quality"] = "pyrrhic"
        data_annotation["line_n"] = attributes["data-n"]
        data_annotation["title"] = title
        data_annotation["author"] = author    
        lines_annotation.append(data_annotation)
    j += 1
    print(f"Poem {j} done")
    time.sleep(1)
    

Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {j} done
Poem {

In [8]:
lines_annotation

[{'line': ' What was he doing, the great god Pan,',
  'met': '--+--+--+--+',
  'real': '+--+--+++',
  'feet': ' What was |he do|ing, the great |god Pan,',
  'feet_number': 'tetrameter',
  'meter_quality': 'anapestic',
  'line_n': '1',
  'title': 'A Musical Instrument(1860)',
  'author': 'Elizabeth Barrett Browning'},
 {'line': 'Down in the reeds by the river?',
  'met': '--+--+--+',
  'real': '+--+--+-',
  'feet': 'Down in |the reeds |by the river?',
  'feet_number': 'trimeter',
  'meter_quality': 'anapestic',
  'line_n': '2',
  'title': 'A Musical Instrument(1860)',
  'author': 'Elizabeth Barrett Browning'},
 {'line': 'Spreading ruin and scattering ban,',
  'met': '--+--+--+--+',
  'real': '+-+--+--+',
  'feet': 'Spreading |ruin |and scat|tering ban,',
  'feet_number': 'tetrameter',
  'meter_quality': 'anapestic',
  'line_n': '3',
  'title': 'A Musical Instrument(1860)',
  'author': 'Elizabeth Barrett Browning'},
 {'line': ' Splashing and paddling with hoofs of a goat,',
  'met': '--+

In [5]:
df = pd.DataFrame(lines_annotation) # converting to dataframe

NameError: name 'lines_annotation' is not defined

In [ ]:
#df.to_csv(r"C:\Users\sacha\Downloads\forbetterverse.csv", index=False, encoding="utf-8")